In [ ]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import pandas as pd
import pyarrow.parquet as pq
from langchain_core.documents import Document
from copy_utils import *
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


# semantic rag pipeline

In [3]:
def build_prompt(SYSTEM_PROMPT, query, context):
    return f"""{SYSTEM_PROMPT}

context:
{context}

question: 
{query}

Answer based on the Amazon datasets: """

def build_context(docs):
    return "\n\n".join(
        f"Product ASIN: {doc.metadata.get('parent_asin', 'N/A')}\n"
        f"Title: {doc.metadata.get('product_title', '')}\n"
        for doc in docs
    )

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
        "../data/processed/langchain_semantic_index", embeddings, allow_dangerous_deserialization=True
    )

# Convert vector store to a retriever
retriever = vectorstore.as_retriever(
search_type="similarity",
search_kwargs={"k": 5} # Fetch 5 most similar documents
)


C:\Users\rocco\AppData\Local\Temp\ipykernel_18240\3867395794.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4163.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B",
    max_new_tokens=128,
    do_sample=True,
)
# Use in a chain
query = "what is a good gaming mouse to buy if I am left handed?"
docs = retriever.invoke(query)
context = build_context(docs)


SYSTEM_PROMPT_1 = """
    You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (which contains real product reviews + metadata).
    Always cite the product ASIN when possible. If the answer isn't in the context, say so.
    """


llm = HuggingFacePipeline(pipeline=generator)
llm_prompt = build_prompt(SYSTEM_PROMPT_1, query, context)
prompt = ChatPromptTemplate.from_template(llm_prompt)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 3782.85it/s]
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [6]:
rag_chain = (
    {
        "context": retriever | RunnableLambda(build_context),
        "query": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [7]:
response = rag_chain.invoke(query)
response_cut = response.split("Assistant:", 1)[-1].strip()

print(response_cut)

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A good gaming mouse to buy if you are left-handed, as per the Amazon datasets, is the 3d Frog USB Wired Optical Mouse.


# bm25 retriever

In [8]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

docs = pd.read_parquet("../data/processed/product_documents.parquet")

# Convert docs dataframe to LangChain Documents
documents = [
    Document(
        page_content=row["document"],
        metadata={"product_title": row["product_title"], "parent_asin": row["parent_asin"], "average_rating": row["average_rating"]}
    )
    for _, row in docs.iterrows()
]

# Build BM25 retriever
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5 

# Search
results = bm25_retriever.invoke("wireless headphones")
for r in results:
    print(r.metadata["product_title"] + str(r.metadata["average_rating"]))

Wireless Bluetooth Headphones Waterproof IPX7, Best Sport in Ear Earbuds Earphones w/Remote and Mic 4.3
Wireless Ear buds Wireless Bluetooth 5.2 Headphones Fast Pairing 3D Stereo Noise Reduction in-Ear Ea3.8
Audio-Technica ATH-A900X Audiophile Closed-Back Dynamic Headphones (Discontinued by Manufacturer)3.7
Logitech 910002698 M525 Wireless Mouse, Compact, Right/Left, Blue4.3
Sony PS-LX310BT Wireless Turntable with Bluetooth Connectivity Bundle with Carbon Fiber Anti-Static 4.6


# hybrid retriever

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Initialize individual retrievers
bm25_retriever = BM25Retriever.from_documents(documents)
vector_retriever = vectorstore.as_retriever()

# Create the ensemble
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]  # Example: asigning 40% importance to BM25, 60% to Semantic Search
)

# Invoke to get combined results
docs = ensemble_retriever.invoke("wireless headphones")
for d in docs:
    print(d.metadata["product_title"])

Wireless Ear buds Wireless Bluetooth 5.2 Headphones Fast Pairing 3D Stereo Noise Reduction in-Ear Ea
Wireless Earbuds,Chisana Bluetooth Headphones Extreme Subwoofer Bass,Bluetooth Earbuds for Stable Co
Sound Isolating Handsfree Headset Earphones Earbuds w Mic Dual Headphones Stereo Flat Wired 3.5mm [B
Wireless Earbuds, Soundmoov Bluetooth Headphones Handsfree Stereo Earphone with Microphone, Noise Ca
Wireless Bluetooth Headphones Waterproof IPX7, Best Sport in Ear Earbuds Earphones w/Remote and Mic 
Wireless Ear buds Wireless Bluetooth 5.2 Headphones Fast Pairing 3D Stereo Noise Reduction in-Ear Ea
Audio-Technica ATH-A900X Audiophile Closed-Back Dynamic Headphones (Discontinued by Manufacturer)
Logitech 910002698 M525 Wireless Mouse, Compact, Right/Left, Blue


: 

# hybrid rag pipeline

In [ ]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B",
    max_new_tokens=128,
    do_sample=True,
)
# Use in a chain
query = "what is a good gaming mouse to buy if I am left handed?"
docs = ensemble_retriever.invoke(query)
context = build_context(docs)


SYSTEM_PROMPT_1 = """
    You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (which contains real product reviews + metadata).
    Always cite the product ASIN when possible. If the answer isn't in the context, say so.
    """


llm = HuggingFacePipeline(pipeline=generator)
llm_prompt = build_prompt(SYSTEM_PROMPT_1, query, context)
prompt = ChatPromptTemplate.from_template(llm_prompt)

In [ ]:
hybrid_rag_chain = (
    {
        "context": ensemble_retriever | RunnableLambda(build_context),
        "query": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = hybrid_rag_chain.invoke(query)
response_cut = response.split("Assistant:", 1)[-1].strip()

print(response_cut)

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on the given context, a good gaming mouse to buy if you are left-handed is the Logitech Anywhere Mobile Mouse MX, Wireless Mouse for PC and Mac.
